# Load the data

In [ ]:
import pandas as pd
messages=pd.read_csv("SMSSpamCollection",sep='\t',names=['label','message'])

# Exploratory Data Analysis (EDA)

In [ ]:
messages.head()

# Group by label and get statistics

In [ ]:
messages.groupby('label').describe()

# Feature Engineering: Add message length as a new feature

In [ ]:
messages['length']=messages['message'].apply(len)

In [ ]:
messages.head()

# Data Visuvalization

In [ ]:
import matplotlib.pyplot as plt
import seaborn

In [ ]:
messages['length'].plot(bins=50,kind='hist')  # Histogram of message lengths


In [ ]:
messages['length'].plot(kind='box')  # Boxplot of lengths by spam/ham


In [ ]:
messages.length.describe()    # Summary statistics of message lengths


# Examine the longest message

In [ ]:
messages[messages['length']==910]['message'].iloc[0]

# Histograms of message lengths for spam and ham


In [ ]:
messages.hist(column='length',by='label',bins=50,figsize=(12,4))

# Text Processing (Cleaning and Preprocessing)

In [ ]:
import nltk

In [ ]:
nltk.download()

In [ ]:
import string

# Test text processing on a sample

In [ ]:
mess="Hi!! i am chandra and i am implementing nltk #chandra"

# Remove punctuation

In [ ]:
nopunc=[char for char in mess if char not in string.punctuation]
nopunc=''.join(nopunc)
nopunc

In [ ]:
from nltk.corpus import stopwords

In [ ]:
stopwords.words('english')[0:20]

In [ ]:
nopunc.split()

# Tokenization and removing stopwords

In [ ]:
clean_mess=[word for word in nopunc.split() if word.lower() not in stopwords.words('english')]
clean_mess

# Text processing function (to be applied to messages)

In [ ]:
def text_process(mess):
    nopunc = [char for char in mess if char not in string.punctuation]
    nopunc = ''.join(nopunc)
    return [word for word in nopunc.split() if word.lower() not in stopwords.words('english')]

# Apply the text processing function on sample message

In [ ]:
text_process('i am chandra i am learning NLP #MachineLearning')

# Apply the text processing function to the first few messages

In [ ]:
messages['message'].head().apply(text_process)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Bag of Words (BoW) Model (Feature Extraction)

In [ ]:
bow_transformer=CountVectorizer(analyzer=text_process).fit(messages['message'])

In [ ]:
len(bow_transformer.vocabulary_)

# Example of transforming a message into BoW representation

In [ ]:
message911=messages['message'][911]
message911

In [ ]:
bow911=bow_transformer.transform([message911])


In [ ]:
print(bow_transformer.get_feature_names_out()[11154])

# Transform all messages into BoW vectors

In [ ]:
message_bow=bow_transformer.transform(messages['message'])

In [ ]:
message_bow.shape

In [ ]:
message_bow.nnz


# Term Frequency-Inverse Document Frequency (TF-IDF) Model

In [ ]:
from sklearn.feature_extraction.text import TfidfTransformer

In [ ]:
tfidf_transformer=TfidfTransformer().fit(message_bow)

# Example of transforming BoW into TF-IDF representation


In [ ]:
tfidf911=tfidf_transformer.transform(bow911)
print(tfidf911)

# Transform all BoW vectors into TF-IDF vectors

In [ ]:
message_tfidf=tfidf_transformer.transform(message_bow)

In [ ]:
message_tfidf.shape

# Naive Bayes Classifier for Spam Detection

In [ ]:
from sklearn.naive_bayes import MultinomialNB

In [ ]:
spam_detector=MultinomialNB().fit(message_tfidf,messages['label'])

# Prediction on the sample message (911)

In [ ]:
print("pridected",spam_detector.predict(tfidf911))

In [ ]:
print("expected",messages.label[911])

# Prediction on the entire dataset

In [ ]:
all_predictions=spam_detector.predict(message_tfidf)

In [ ]:
from sklearn.metrics import classification_report,confusion_matrix

# Model Evaluation

In [ ]:
print(classification_report(messages['label'],all_predictions))

# Train-Test Split for Model Evaluation (Using Pipelines)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
msg_train,msg_test,label_train,label_test=train_test_split(messages['message'],messages['label'],test_size=0.2)

In [ ]:
len(msg_test)

In [ ]:
len(msg_train)

In [ ]:
from sklearn.pipeline import Pipeline

# Pipeline for text processing, feature extraction, and classification

In [ ]:
pipeline=Pipeline([
    ('bow',CountVectorizer(analyzer=text_process)),
    ('tfidf',TfidfTransformer()),
    ('classifier',MultinomialNB())
])

# Train the pipeline on the training data

In [ ]:
pipeline.fit(X=msg_train,y=label_train)

# Make predictions on the test data

In [ ]:
predictions=pipeline.predict(msg_test)


# Evaluate the pipeline

In [ ]:
print(classification_report(predictions,label_test))

In [ ]:
print(confusion_matrix(predictions,label_test))